# Notebook 02 — 全 A 股财务数据清洗

**输入**：`data/raw/financials_2021_2023.xlsx`（iFind 导出，全 A 股 2021-2023 年报，约 5000 家）  
**输出**：
- `data/processed/02_financials_long.csv`（长表，主用，公司 × 年份）
- `data/processed/02_financials_wide.csv`（宽表备份）

## 清洗流程
| Step | 内容 | 关键点 |
|------|------|--------|
| 1 | 读取原始 xlsx | 处理 iFind 样式兼容问题 |
| 2 | 重命名所有列 | 超长列名 → 简短英文 |
| 3 | 数据清洗 | 去千位符、转数值、剔除北交所/金融业 |
| 4 | 宽表 → 长表 | 78 指标列拆成 年份 × 26 指标 |
| 5 | 审计意见编码 | 文字 → 0/1/2/3/4 |
| 6 | 缺失值检查 | 只看不填 |
| 7 | 保存输出 | 打印质量报告 |

In [36]:
# 安装依赖（第一次运行时执行，之后可跳过）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas", "openpyxl", "python-calamine", "numpy", "-q"])


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


CompletedProcess(args=['/usr/local/bin/python3', '-m', 'pip', 'install', 'pandas', 'openpyxl', 'python-calamine', 'numpy', '-q'], returncode=0)

In [37]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ── 路径配置 ──────────────────────────────────────────────────────
PROJECT_ROOT  = Path("..").resolve()
RAW_FILE      = PROJECT_ROOT / "data" / "raw" / "financials_2021_2023.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"原始文件路径: {RAW_FILE}")
print(f"文件存在: {RAW_FILE.exists()}")

原始文件路径: /Users/aa00551/Desktop/Projects/A-Share-ST-Risk-Predictor/data/raw/financials_2021_2023.xlsx
文件存在: True


## Step 1 — 读取原始 xlsx

iFind 导出的 Excel 有已知样式兼容问题（Colors must be aRGB hex values），按优先级尝试三种读法：
1. 默认引擎（openpyxl）
2. `calamine` 引擎（绕过样式解析）
3. zipfile + xml 手动解析（最稳但最慢）

iFind 的财务宽表第 1 行直接就是字段名（不是双层表头），默认参数即可。

In [38]:
def read_ifind_financial_excel(file_path: Path) -> pd.DataFrame:
    """读取 iFind 财务宽表，自动处理样式兼容问题。"""

    # 方法一：默认引擎
    try:
        df = pd.read_excel(file_path)
        print("✓ 默认引擎（openpyxl）读取成功")
        return df
    except Exception as e1:
        print(f"默认引擎失败: {e1}")

    # 方法二：calamine 引擎（跳过样式解析，速度也更快）
    try:
        df = pd.read_excel(file_path, engine="calamine")
        print("✓ calamine 引擎读取成功")
        return df
    except Exception as e2:
        print(f"calamine 引擎失败: {e2}")

    # 方法三：zipfile + xml 手动解析（兜底方案）
    print("→ 尝试 zipfile + xml 手动解析...")
    import zipfile
    import xml.etree.ElementTree as ET

    with zipfile.ZipFile(file_path) as zf:
        # 读取共享字符串表
        shared_strings = []
        if "xl/sharedStrings.xml" in zf.namelist():
            tree = ET.parse(zf.open("xl/sharedStrings.xml"))
            ns = {"ns": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
            for si in tree.findall(".//ns:si", ns):
                texts = [t.text or "" for t in si.findall(".//ns:t", ns)]
                shared_strings.append("".join(texts))

        # 读取 sheet1
        tree = ET.parse(zf.open("xl/worksheets/sheet1.xml"))
        ns = {"ns": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
        rows_data = []
        for row in tree.findall(".//ns:row", ns):
            row_vals = []
            for cell in row.findall("ns:c", ns):
                cell_type = cell.get("t", "")
                val_el = cell.find("ns:v", ns)
                if val_el is None:
                    row_vals.append(None)
                elif cell_type == "s":  # 共享字符串索引
                    row_vals.append(shared_strings[int(val_el.text)])
                else:
                    row_vals.append(val_el.text)
            rows_data.append(row_vals)

    if not rows_data:
        raise ValueError("zipfile 解析失败，rows_data 为空")

    # 第一行作为列名，剩余行作为数据
    max_cols = max(len(r) for r in rows_data)
    headers = rows_data[0] + [None] * (max_cols - len(rows_data[0]))
    data_rows = [r + [None] * (max_cols - len(r)) for r in rows_data[1:]]
    df = pd.DataFrame(data_rows, columns=headers)
    print("✓ zipfile + xml 手动解析成功")
    return df


df_raw = read_ifind_financial_excel(RAW_FILE)

print(f"\n总行数: {len(df_raw)}（预期 5000+）")
print(f"总列数: {len(df_raw.columns)}（预期约 84 列）")
print("\n列名预览（前 10 列）：")
for i, col in enumerate(df_raw.columns[:10]):
    print(f"  [{i}] {repr(col)}")
print("\n前 3 行预览（只显示前 6 列）：")
df_raw.iloc[:3, :6]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✓ 默认引擎（openpyxl）读取成功

总行数: 5514（预期 5000+）
总列数: 97（预期约 84 列）

列名预览（前 10 列）：
  [0] '证券代码'
  [1] '证券名称'
  [2] '股票代码'
  [3] '股票简称'
  [4] '所属申万行业\n[行业级别] 二级行业\n[截止日期] 20251231'
  [5] '首发上市日期'
  [6] '总资产\n[报告期] 2021年报\n[报表类型] 合并报表\n[货币币种] 原始币种\n[单位]  元'
  [7] '总资产\n[报告期] 2022年报\n[报表类型] 合并报表\n[货币币种] 原始币种\n[单位]  元'
  [8] '总资产\n[报告期] 2023年报\n[报表类型] 合并报表\n[货币币种] 原始币种\n[单位]  元'
  [9] '流动比率\n[报告期] 2021年报'

前 3 行预览（只显示前 6 列）：


,证券代码,证券名称,股票代码,股票简称,所属申万行业\n[行业级别] 二级行业\n[截止日期] 20251231,首发上市日期
0,920000.BJ,安徽凤凰,920000.0,安徽凤凰,汽车零部件,20201223.0
1,920001.BJ,纬达光电,920001.0,纬达光电,光学光电子,20221227.0
2,920002.BJ,万达轴承,920002.0,万达轴承,通用设备,20240530.0


## Step 2 — 重命名所有列（关键步骤！）

iFind 列名含换行符且超长，例如：
```
"总资产\n[报告期] 2021年报\n[报表类型] 合并报表\n[货币币种] 原始币种\n[单位]  元"
```

策略：取列名第一行（换行前的部分）作为指标名，用正则提取年份，通过映射字典转成英文。

**跑完后必须检查是否有 `unknown_xxx` 兜底命名**，有的话停下来告知用户补充映射。

In [39]:
# 指标名（中文）→ 英文 映射字典
INDICATOR_MAP = {
    '总资产':                         'total_assets',
    '流动比率':                        'current_ratio',
    '速动比率':                        'quick_ratio',
    '资产负债率':                      'debt_to_asset',
    '产权比率':                        'debt_to_equity',
    '现金流量利息保障倍数':              'cash_interest_coverage',
    '净资产收益率同比增长(ROE)':         'roe',
    '净资产收益率同比增长（ROE）':        'roe',   # 全角括号兼容
    '总资产报酬率ROA':                  'roa',
    '销售毛利率':                      'gross_margin',
    '销售净利率':                      'net_margin',
    '营业利润/营业总收入':              'operating_margin',
    '每股收益EPS-基本':                 'eps',
    '总资产周转率':                    'asset_turnover',
    '应收账款周转率':                   'ar_turnover',
    '存货周转率':                      'inventory_turnover',
    '流动资产周转率':                   'current_asset_turnover',
    '固定资产周转率':                   'fixed_asset_turnover',
    '营业总收入(同比增长率)':            'revenue_growth',
    '归属母公司股东的净利润(同比增长率)': 'net_profit_growth',
    '资产总计(相对年初增长率)':          'total_assets_growth',
    '净资产(同比增长率)':               'net_assets_growth',
    '经营活动产生的现金流量净额':         'operating_cash_flow',
    '经营活动净收益／利润总额':          'operating_income_ratio',
    '每股经营现金净流量':               'ocf_per_share',
    '营运资本':                        'working_capital',
    '留存收益':                        'retained_earnings',
    '息税前利润EBIT':                   'ebit',
    '营业总收入':                      'total_revenue',
    '审计意见类型':                    'audit_opinion',
    '前十大股东持股比例合计':            'top10_holders_pct',
}


def rename_column(old_col: str) -> str:
    """把 iFind 超长列名转成简短英文列名。"""
    old_col_str = str(old_col)

    # 提取年份（如 2021/2022/2023）
    year_match = re.search(r'(\d{4})年报', old_col_str)
    year = year_match.group(1) if year_match else None

    # 取列名第一行（换行前的内容）作为指标名
    first_line = old_col_str.split('\n')[0].strip()

    # 无年份 → 基础信息列
    if year is None:
        if first_line in ('证券代码', '股票代码', '代码'):  return 'stock_code'
        if first_line in ('证券名称', '股票简称', '名称'):  return 'stock_name'
        if '所属申万行业' in old_col_str:                   return 'sw_industry'
        if first_line == '首发上市日期':                    return 'ipo_date'
        if '公司性质' in old_col_str:                       return 'company_type'
        # 兜底
        return f'unknown_base_{re.sub(r"[^\w]", "", first_line)[:12]}'

    # 有年份 → 指标列
    for cn, en in INDICATOR_MAP.items():
        if first_line == cn:
            return f'{en}_{year}'

    # 兜底（跑完后必须检查！）
    return f'unknown_{re.sub(r"[^\w]", "", first_line)[:12]}_{year}'


# 重命名
new_cols = [rename_column(c) for c in df_raw.columns]
df_raw.columns = new_cols

print("重命名后所有列名：")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i:02d}] {col}")

# ── 关键检查：有无 unknown_xxx 兜底命名 ──────────────────────────
unknown_cols = [c for c in df_raw.columns if c.startswith('unknown_')]
if unknown_cols:
    print(f"\n⚠️  发现 {len(unknown_cols)} 个未识别列，请告知用户补充映射：")
    for c in unknown_cols:
        print(f"   → {c}")
    raise ValueError("存在 unknown_xxx 列，停止执行，请补充 INDICATOR_MAP 映射后重跑")
else:
    print(f"\n✓ 所有 {len(df_raw.columns)} 列已成功映射，无兜底命名")

重命名后所有列名：
  [00] stock_code
  [01] stock_name
  [02] stock_code
  [03] stock_name
  [04] sw_industry
  [05] ipo_date
  [06] total_assets_2021
  [07] total_assets_2022
  [08] total_assets_2023
  [09] current_ratio_2021
  [10] current_ratio_2022
  [11] current_ratio_2023
  [12] quick_ratio_2021
  [13] quick_ratio_2022
  [14] quick_ratio_2023
  [15] debt_to_asset_2021
  [16] debt_to_asset_2022
  [17] debt_to_asset_2023
  [18] debt_to_equity_2021
  [19] debt_to_equity_2022
  [20] debt_to_equity_2023
  [21] cash_interest_coverage_2021
  [22] cash_interest_coverage_2022
  [23] cash_interest_coverage_2023
  [24] roe_2021
  [25] roe_2022
  [26] roe_2023
  [27] roa_2021
  [28] roa_2022
  [29] roa_2023
  [30] gross_margin_2021
  [31] gross_margin_2022
  [32] gross_margin_2023
  [33] net_margin_2021
  [34] net_margin_2022
  [35] net_margin_2023
  [36] operating_margin_2021
  [37] operating_margin_2022
  [38] operating_margin_2023
  [39] eps_2021
  [40] eps_2022
  [41] eps_2023
  [42] asset_turnov

## Step 3 — 数据清洗

依次完成：
- **3.1** 删除重复列（如 `stock_code_dup`）
- **3.2** 去千位分隔符 + 转数值（`655,571,748.33` → `float`）
- **3.3** 转换 `ipo_date` 为 datetime
- **3.4** 剔除北交所（`.BJ`）
- **3.5** 剔除金融业

In [40]:
# ── 3.1 删除重复/无用列 ───────────────────────────────────────────
dup_cols = ['stock_code_dup', 'stock_name_dup']
df = df_raw.drop(columns=dup_cols, errors='ignore').copy()
print(f"3.1 删除重复列后: {len(df.columns)} 列")

3.1 删除重复列后: 97 列


In [41]:
# ── 3.2 处理千位分隔符 + 转数值 ──────────────────────────────────
# 不参与数值转换的列（文本/日期类）
non_numeric_cols = ['stock_code', 'stock_name', 'sw_industry', 'ipo_date', 'company_type']
audit_cols       = [c for c in df.columns if c.startswith('audit_opinion')]
str_cols         = set(non_numeric_cols + audit_cols)

numeric_cols = [c for c in df.columns if c not in str_cols]

convert_errors = {}  # 记录转换时报错的列，供排查

for col in numeric_cols:
    if df[col].dtype == object:
        sample_before = df[col].dropna().head(3).tolist()
        df[col] = (
            df[col].astype(str)
            .str.replace(',', '', regex=False)  # 去千位符
            .str.strip()
            .replace({'': np.nan, 'nan': np.nan, '--': np.nan, 'None': np.nan})
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

        # 如果全部变 NaN 且原来有值，打印警告
        if df[col].isna().all() and len(sample_before) > 0:
            convert_errors[col] = sample_before

if convert_errors:
    print("⚠️  以下列转换后全为 NaN，请检查原始值格式：")
    for col, samples in convert_errors.items():
        print(f"  {col}: 原始值示例 = {samples}")
else:
    print(f"✓ 3.2 数值转换完成，共处理 {len(numeric_cols)} 个数值列")

✓ 3.2 数值转换完成，共处理 87 个数值列


In [42]:
# ── 3.3 转换 ipo_date 为 datetime ────────────────────────────────
# iFind 可能导出 "20201223" 这种 8 位字符串，也可能已是 datetime
df['ipo_date'] = pd.to_datetime(
    df['ipo_date'].astype(str).str.replace('-', ''),
    format='%Y%m%d',
    errors='coerce'
)

valid_ipo = df['ipo_date'].notna().sum()
print(f"✓ 3.3 ipo_date 转换完成: {valid_ipo}/{len(df)} 行有效")
print(f"   上市日期范围: {df['ipo_date'].min().date()} ~ {df['ipo_date'].max().date()}")

✓ 3.3 ipo_date 转换完成: 0/5514 行有效
   上市日期范围: NaT ~ NaT


In [43]:
# ── 3.4 剔除北交所（代码后缀 .BJ）────────────────────────────────
# 先强制去重，防止旧版 Step 2 把「证券代码」和「股票代码」都映射成 stock_code
df = df.loc[:, ~df.columns.duplicated(keep='first')]

before = len(df)
df = df[~df['stock_code'].astype(str).str.endswith('.BJ')].copy()
print(f'3.4 剔除北交所后: {len(df)} 家（剔除 {before - len(df)} 家）')

3.4 剔除北交所后: 5203 家（剔除 311 家）


In [44]:
# ── 3.5 剔除金融业 ───────────────────────────────────────────────
# 用申万行业（sw_industry）识别金融，iFind 申万行业更细
finance_keywords = ['银行', '保险', '证券', '多元金融']
mask_finance = df['sw_industry'].astype(str).str.contains(
    '|'.join(finance_keywords), na=False
)

before = len(df)
print(f"被识别为金融业的公司: {mask_finance.sum()} 家")
if mask_finance.sum() > 0:
    print(df[mask_finance][['stock_code', 'stock_name', 'sw_industry']].head(10).to_string(index=False))

df = df[~mask_finance].copy()
print(f"3.5 剔除金融业后: {len(df)} 家公司（剔除 {before - len(df)} 家）")
print(f"\n🔢 宽表最终规模: {len(df)} 行 × {len(df.columns)} 列")

被识别为金融业的公司: 97 家
stock_code stock_name sw_industry
 600000.SH       浦发银行      股份制银行Ⅱ
 600015.SH       华夏银行      股份制银行Ⅱ
 600016.SH       民生银行      股份制银行Ⅱ
 600030.SH       中信证券         证券Ⅱ
 600036.SH       招商银行      股份制银行Ⅱ
 600053.SH      *ST九鼎        多元金融
 600061.SH       国投资本         证券Ⅱ
 600095.SH       湘财股份         证券Ⅱ
 600109.SH       国金证券         证券Ⅱ
 600120.SH       浙江东方        多元金融
3.5 剔除金融业后: 5106 家公司（剔除 97 家）

🔢 宽表最终规模: 5106 行 × 95 列


## Step 4 — 宽表 → 长表

把 `total_assets_2021` / `total_assets_2022` / `total_assets_2023` 这种结构，
转成一列 `year` + 一列 `total_assets` 的长表格式。

`pd.wide_to_long` 会识别 `stubnames`（指标名）+ `sep='_'` + `suffix='\d+'`（年份数字），自动完成转换。

预期：宽表 N 行 → 长表 N × 3 行（2021/2022/2023）

In [45]:
# 找出所有"指标_年份"形式的列
indicator_cols = [c for c in df.columns if re.search(r'_(2021|2022|2023)$', c)]

# 提取所有 stub 名（去掉 _年份 后缀）
stub_names = sorted(set([re.sub(r'_(2021|2022|2023)$', '', c) for c in indicator_cols]))

print(f"指标列总数: {len(indicator_cols)}")
print(f"唯一指标数: {len(stub_names)}（预期约 26）")
print(f"指标列表: {stub_names}")

# 基础信息列（不随年份变化）
id_cols_all = ['stock_code', 'stock_name', 'sw_industry', 'ipo_date', 'company_type']
id_cols = [c for c in id_cols_all if c in df.columns]  # 防止某列不存在
print(f"\n基础信息列: {id_cols}")

# ── wide_to_long 转换 ─────────────────────────────────────────────
df_long = pd.wide_to_long(
    df,
    stubnames=stub_names,
    i=id_cols,
    j='year',
    sep='_',
    suffix=r'\d+'
).reset_index()

df_long['year'] = df_long['year'].astype(int)

print(f"\n长表行数: {len(df_long)}（预期 ≈ {len(df)} × 3 = {len(df)*3}）")
print(f"长表列数: {len(df_long.columns)}（预期 ≈ {len(id_cols)} 基础 + 1 year + {len(stub_names)} 指标 = {len(id_cols)+1+len(stub_names)}）")

print("\n年份分布：")
print(df_long['year'].value_counts().sort_index())

print("\n前 5 行预览：")
df_long.head()

指标列总数: 90
唯一指标数: 30（预期约 26）
指标列表: ['ar_turnover', 'asset_turnover', 'audit_opinion', 'cash_interest_coverage', 'current_asset_turnover', 'current_ratio', 'debt_to_asset', 'debt_to_equity', 'ebit', 'eps', 'fixed_asset_turnover', 'gross_margin', 'inventory_turnover', 'net_assets_growth', 'net_margin', 'net_profit_growth', 'ocf_per_share', 'operating_cash_flow', 'operating_income_ratio', 'operating_margin', 'quick_ratio', 'retained_earnings', 'revenue_growth', 'roa', 'roe', 'top10_holders_pct', 'total_assets', 'total_assets_growth', 'total_revenue', 'working_capital']

基础信息列: ['stock_code', 'stock_name', 'sw_industry', 'ipo_date', 'company_type']

长表行数: 15318（预期 ≈ 5106 × 3 = 15318）
长表列数: 36（预期 ≈ 5 基础 + 1 year + 30 指标 = 36）

年份分布：
year
2021    5106
2022    5106
2023    5106
Name: count, dtype: int64

前 5 行预览：


,stock_code,stock_name,sw_industry,ipo_date,company_type,year,ar_turnover,asset_turnover,audit_opinion,cash_interest_coverage,...,quick_ratio,retained_earnings,revenue_growth,roa,roe,top10_holders_pct,total_assets,total_assets_growth,total_revenue,working_capital
0,600004.SH,白云机场,航空机场,NaT,省属国资控股,2021,4.3539,0.1924,标准无保留意见,683.699853,...,0.4750,5.972225e+09,-0.8498,-1.6539,-0.7523,65.82,2.754503e+10,4.7408,5.180238e+09,-2.701969e+09
1,600004.SH,白云机场,航空机场,NaT,省属国资控股,2022,3.6528,0.1458,标准无保留意见,629.421544,...,0.4605,4.900391e+09,-23.3441,-4.9207,-3.8741,63.89,2.694079e+10,-2.1937,3.970960e+09,-3.431764e+09
2,600004.SH,白云机场,航空机场,NaT,省属国资控股,2023,6.4324,0.2421,标准无保留意见,1730.413018,...,0.6259,5.341796e+09,61.9475,2.8776,8.5752,63.88,2.619005e+10,-2.7866,6.430868e+09,-1.878313e+09
3,600006.SH,东风股份,商用车,NaT,央企国资控股,2021,3.7278,0.7766,标准无保留意见,10677.003989,...,1.1747,5.390383e+09,13.2279,1.3287,-2.5772,61.62,1.988165e+10,-1.3954,1.555004e+10,4.418649e+09
4,600006.SH,东风股份,商用车,NaT,央企国资控股,2022,3.3324,0.6490,标准无保留意见,-3205.712603,...,1.3595,5.563867e+09,-21.6080,0.6879,-1.1787,61.79,1.768608e+10,-11.0432,1.218999e+10,4.986055e+09


## Step 5 — 审计意见编码

将文字型审计意见转成有序数值编码，方便建模使用：

| 原文 | 编码 | 含义 |
|------|------|------|
| 标准无保留意见 | 0 | 正常 |
| 带强调事项段的无保留意见 | 1 | 轻微关注 |
| 保留意见 | 2 | 有问题 |
| 否定意见 | 3 | 严重问题 |
| 无法表示意见 | 4 | 最严重 |

In [46]:
audit_mapping = {
    '标准无保留意见':             0,
    '带强调事项段的无保留意见':    1,
    '保留意见':                   2,
    '否定意见':                   3,
    '无法表示意见':               4,
}

df_long['audit_opinion_code']  = df_long['audit_opinion'].map(audit_mapping)
df_long['is_non_standard_audit'] = (df_long['audit_opinion_code'] > 0).astype(int)

print('审计意见分布：')
print(df_long['audit_opinion'].value_counts(dropna=False))

print('\n编码分布：')
print(df_long['audit_opinion_code'].value_counts(dropna=False).sort_index())

print('\n是否非标审计：')
print(df_long['is_non_standard_audit'].value_counts())

# 检查是否有审计意见未被映射（audit_opinion_code 为 NaN 但 audit_opinion 不为空）
unmapped = df_long[
    df_long['audit_opinion_code'].isna() & df_long['audit_opinion'].notna()
][['stock_code', 'year', 'audit_opinion']]

if len(unmapped) > 0:
    print(f"\n⚠️  发现 {len(unmapped)} 条未映射的审计意见，请补充 audit_mapping：")
    print(unmapped['audit_opinion'].value_counts())
else:
    print("\n✓ 所有非空审计意见均已成功映射")

审计意见分布：
audit_opinion
标准无保留意见         14768
带强调事项段的无保留意见      245
保留意见              196
NaN                89
无法表示意见             20
Name: count, dtype: int64

编码分布：
audit_opinion_code
0.0    14768
1.0      245
2.0      196
4.0       20
NaN       89
Name: count, dtype: int64

是否非标审计：
is_non_standard_audit
0    14857
1      461
Name: count, dtype: int64

✓ 所有非空审计意见均已成功映射


## Step 6 — 缺失值检查（只看不填）

先统计各字段缺失率，**不做填充**。后续建模阶段根据字段重要性和缺失原因再决定填充策略。

缺失率高的常见原因：
- 某些指标对特定行业不适用（如存货周转率对服务业）
- 公司规模小，部分指标未披露
- iFind 数据源本身有缺漏

In [47]:
# 各列缺失率（降序排列）
missing_pct = (df_long.isna().sum() / len(df_long) * 100).round(2).sort_values(ascending=False)

print('缺失率 Top 15：')
print(missing_pct.head(15).to_string())

print('\n缺失率为 0 的列（数据完整）：')
complete_cols = missing_pct[missing_pct == 0].index.tolist()
print(complete_cols)

print(f'\n缺失率 > 20% 的列共 {(missing_pct > 20).sum()} 个')
print(f'缺失率 > 50% 的列共 {(missing_pct > 50).sum()} 个（建模时需重点决策）')

缺失率 Top 15：
ipo_date                  100.00
operating_income_ratio     17.74
top10_holders_pct           7.92
cash_interest_coverage      4.22
inventory_turnover          1.91
ar_turnover                 1.12
roe                         1.02
gross_margin                0.80
fixed_asset_turnover        0.78
roa                         0.74
working_capital             0.74
ebit                        0.74
current_ratio               0.74
current_asset_turnover      0.74
quick_ratio                 0.74

缺失率为 0 的列（数据完整）：
['year', 'is_non_standard_audit']

缺失率 > 20% 的列共 1 个
缺失率 > 50% 的列共 1 个（建模时需重点决策）


## Step 7 — 保存输出 + 质量报告

In [48]:
# ── 保存长表（主用） ──────────────────────────────────────────────
long_path = PROCESSED_DIR / '02_financials_long.csv'
df_long.to_csv(long_path, index=False, encoding='utf-8-sig')
print(f'✅ 长表已保存: {long_path}')
print(f'   规模: {len(df_long)} 行 × {len(df_long.columns)} 列')

# ── 质量报告 ─────────────────────────────────────────────────────
year_counts    = df_long['year'].value_counts().sort_index()
non_std_rate   = df_long['is_non_standard_audit'].mean() * 100
top3_missing   = missing_pct.head(3)

print()
print('=' * 56)
print('         财务数据质量报告')
print('=' * 56)
print(f'全 A 股公司数（剔除金融业、北交所后）: {len(df):,}')
print(f'长表总行数（公司 × 3 年）:            {len(df_long):,}')
print(f'年份分布:  ', end='')
print('  '.join([f"{yr}: {cnt}" for yr, cnt in year_counts.items()]))
print(f'非标审计意见占比:   {non_std_rate:.2f}%')
print('缺失率最高字段 Top 3：')
for col, pct in top3_missing.items():
    print(f'  - {col}: {pct}%')
print('=' * 56)

✅ 长表已保存: /Users/aa00551/Desktop/Projects/A-Share-ST-Risk-Predictor/data/processed/02_financials_long.csv
   规模: 15318 行 × 38 列

         财务数据质量报告
全 A 股公司数（剔除金融业、北交所后）: 5,106
长表总行数（公司 × 3 年）:            15,318
年份分布:  2021: 5106  2022: 5106  2023: 5106
非标审计意见占比:   3.01%
缺失率最高字段 Top 3：
  - ipo_date: 100.0%
  - operating_income_ratio: 17.74%
  - top10_holders_pct: 7.92%
